
# Reto Bootcamp: Segmentación Inteligente de Clientes en Retail Online

## Contexto del reto

Una empresa de retail online necesita entender mejor a sus clientes: quiénes compran más, con qué frecuencia, y cuánto valor le aportan al negocio. En vez de tratar a todos los clientes por igual, el objetivo es **segmentarlos en grupos con comportamientos similares**, para poder diseñar estrategias comerciales diferenciadas (por ejemplo, retener a los clientes de alto valor o reactivar a los inactivos).

Para lograrlo, este notebook sigue estas fases:

1. Comprensión del problema (conceptual)
2. Análisis exploratorio de datos (EDA)
3. Preparación y transformación de datos
4. Ingeniería de variables — modelo RFM (Recency, Frequency, Monetary)
5. Segmentación con aprendizaje no supervisado (K-Means)
6. Interpretación de los segmentos
7. Modelo supervisado explicativo (árbol de decisión)
8. Conclusiones

**Dataset:** *Online Retail* — UCI Machine Learning Repository. Transacciones de una tienda online del Reino Unido entre 2010 y 2011, con más de 500.000 registros.

En cada bloque de código encontrarás primero el **contexto** (qué vamos a hacer y por qué), luego el **código**, y después, en una celda aparte, la **explicación de los resultados**.



## Fase 1: Comprensión del problema

Esta fase es conceptual — el reto pide reflexión, no código.

**¿Qué significa que un cliente sea valioso para un negocio de retail?**
Un cliente valioso no es solo el que compra más caro una vez, sino el que genera ingresos de forma sostenida en el tiempo: compra con cierta frecuencia, mantiene una relación activa con la marca y aporta un volumen de ingresos significativo frente al resto de la base de clientes.

**¿Por qué no todos los clientes deben tratarse de la misma manera?**
Porque sus comportamientos y su potencial de negocio son distintos. Tratar igual a un cliente ocasional que a uno frecuente y de alto gasto desperdicia recursos de marketing y puede hacer que la empresa pierda a sus clientes más importantes por falta de atención diferenciada.

**¿Qué tipo de decisiones podría apoyar una buena segmentación?**
Campañas de retención para clientes en riesgo de abandono, programas de fidelización para clientes de alto valor, estrategias de reactivación para clientes inactivos, y priorización de recursos de atención al cliente según el segmento.



## Fase 2: Análisis Exploratorio de Datos (EDA)

### Importar librerías y cargar los datos

Cargamos el dataset *Online Retail* directamente desde la URL de la UCI. El archivo viene en formato Excel (.xlsx), así que usamos `pd.read_excel`.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
datos = pd.read_excel(url)

print("Forma del dataset:", datos.shape)
datos.head()



deberías ver alrededor de 541.909 filas y 8 columnas (`InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`). Cada fila representa un producto dentro de una factura, no una factura completa — un mismo `InvoiceNo` puede repetirse en varias filas. La carga puede tardar uno o dos minutos porque el archivo es grande.


### Inspección de la estructura del dataset

Revisamos los tipos de datos de cada columna y cuántos valores únicos hay en las columnas clave.


In [ ]:

datos.info()


In [ ]:

print("Facturas únicas:", datos["InvoiceNo"].nunique())
print("Clientes únicos:", datos["CustomerID"].nunique())
print("Países:", datos["Country"].nunique())
print("Rango de fechas:", datos["InvoiceDate"].min(), "a", datos["InvoiceDate"].max())



`CustomerID` debería mostrarse como número decimal (float), no entero, porque tiene valores faltantes (pandas convierte automáticamente una columna numérica con nulos a float). Vas a ver más de 4.000 clientes únicos, cerca de 25.000 facturas, y datos concentrados sobre todo en Reino Unido, cubriendo aproximadamente un año (diciembre de 2010 a diciembre de 2011).


### Identificación de valores faltantes

`CustomerID` es la columna más crítica para este reto, porque sin ella no podemos calcular el comportamiento de un cliente.


In [ ]:

datos.isnull().sum()



normalmente `CustomerID` tiene una cantidad considerable de valores nulos (unas 135.000 filas, cerca del 25% del dataset) — son ventas registradas sin identificar al cliente. `Description` también suele tener algunos pocos nulos. Como el reto se centra en el comportamiento *por cliente*, más adelante vamos a descartar las filas sin `CustomerID`, porque no se pueden asociar a ningún segmento.


### Análisis estadístico descriptivo

Revisamos los estadísticos básicos de las columnas numéricas: `Quantity` y `UnitPrice`.


In [ ]:

datos[["Quantity", "UnitPrice"]].describe()



es muy probable que veas valores mínimos **negativos** tanto en `Quantity` como en `UnitPrice`. Los valores negativos en `Quantity` corresponden a devoluciones (facturas cuyo `InvoiceNo` empieza con la letra "C" de "cancelación"), y los valores negativos o en cero en `UnitPrice` suelen ser ajustes contables, no ventas reales. También vas a notar una diferencia enorme entre la mediana y el máximo, lo que confirma que hay valores atípicos (outliers) importantes — compras corporativas muy grandes frente a compras normales de un solo cliente.


### Visualización de la distribución de `Quantity` y `UnitPrice`

Como ya sabemos que hay outliers extremos, limitamos el rango visual del histograma para poder ver la forma real de la distribución.


In [ ]:

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))

datos[datos["Quantity"].between(0, 50)]["Quantity"].hist(bins=30, ax=ejes[0])
ejes[0].set_title("Distribución de Quantity (0 a 50)")
ejes[0].set_xlabel("Cantidad")

datos[datos["UnitPrice"].between(0, 20)]["UnitPrice"].hist(bins=30, ax=ejes[1])
ejes[1].set_title("Distribución de UnitPrice (0 a 20)")
ejes[1].set_xlabel("Precio unitario")

plt.tight_layout()
plt.show()



ambas distribuciones deberían verse muy sesgadas hacia la izquierda (asimetría positiva): la mayoría de las compras son de pocas unidades y precios bajos, con una cola larga de compras grandes o productos caros. Esto es típico en datos de retail y confirma por qué, más adelante, conviene revisar bien los outliers antes de calcular las variables RFM — unos pocos valores extremos pueden distorsionar el promedio de un cliente.


### Identificación de outliers y cancelaciones

Revisamos cuántas filas corresponden a cancelaciones (facturas que empiezan con "C") y cuántas tienen cantidades o precios negativos o en cero.


In [ ]:

cancelaciones = datos["InvoiceNo"].astype(str).str.startswith("C")
print("Filas de cancelaciones:", cancelaciones.sum())

print("Filas con Quantity <= 0:", (datos["Quantity"] <= 0).sum())
print("Filas con UnitPrice <= 0:", (datos["UnitPrice"] <= 0).sum())



vas a ver varios miles de filas de cancelaciones (típicamente entre 8.000 y 9.000) y una cantidad similar de filas con `Quantity` o `UnitPrice` menores o iguales a cero. Estas filas no representan compras reales, así que en la siguiente fase las vamos a filtrar para no inflar ni distorsionar el cálculo del comportamiento de cada cliente.


## Fase 3: Preparación y transformación de datos

### Filtrar registros irrelevantes o incompletos

Vamos a quedarnos solo con transacciones válidas:

- Sin cancelaciones (`InvoiceNo` que no empiece con "C")
- `Quantity` mayor a 0
- `UnitPrice` mayor a 0
- `CustomerID` no nulo


In [ ]:

datos_limpios = datos[
    (~datos["InvoiceNo"].astype(str).str.startswith("C")) &
    (datos["Quantity"] > 0) &
    (datos["UnitPrice"] > 0) &
    (datos["CustomerID"].notnull())
].copy()

print("Filas antes de filtrar:", datos.shape[0])
print("Filas después de filtrar:", datos_limpios.shape[0])



deberíamos quedarnos con alrededor de 390.000 a 400.000 filas, cerca del 72% de los datos originales. Es una reducción considerable, pero necesaria: nos aseguramos de trabajar solo con compras reales de clientes identificados, que es justamente lo que necesitamos para calcular el comportamiento de cada uno.


### Calcular el valor monetario de cada transacción

Cada fila representa la compra de un producto. El valor de esa línea es `Quantity * UnitPrice`.


In [ ]:

datos_limpios["ValorTotal"] = datos_limpios["Quantity"] * datos_limpios["UnitPrice"]

datos_limpios[["InvoiceNo", "CustomerID", "Quantity", "UnitPrice", "ValorTotal"]].head()



ahora cada fila tiene una columna `ValorTotal` con el importe real de esa línea de compra. Esta es la columna que vamos a sumar por cliente para obtener la variable *Monetary* del modelo RFM.


### Convertir la fecha a un formato adecuado

Nos aseguramos de que `InvoiceDate` sea un tipo de dato fecha, para poder calcular cuánto tiempo ha pasado desde la última compra de cada cliente.


In [ ]:

datos_limpios["InvoiceDate"] = pd.to_datetime(datos_limpios["InvoiceDate"])

print(datos_limpios["InvoiceDate"].dtype)
print("Primera fecha:", datos_limpios["InvoiceDate"].min())
print("Última fecha:", datos_limpios["InvoiceDate"].max())



`InvoiceDate` debería quedar como `datetime64`, y las fechas mínima y máxima deberían coincidir con el rango que vimos en el EDA (diciembre 2010 a diciembre 2011). Con la fecha en este formato ya podemos calcular diferencias de días, que es justo lo que necesita la variable *Recency*.


## Fase 4: Ingeniería de variables — Modelo RFM

El modelo RFM resume el comportamiento de cada cliente en tres números:

- **Recency (R):** cuántos días han pasado desde la última compra del cliente. Un número más bajo es mejor (compró hace poco).
- **Frequency (F):** cuántas facturas distintas ha generado el cliente. Un número más alto es mejor.
- **Monetary (M):** cuánto ha gastado en total el cliente. Un número más alto es mejor.

Para calcular *Recency* necesitamos una fecha de referencia. Usamos el día siguiente a la última fecha del dataset, para que ningún cliente tenga Recency = 0.


In [ ]:

fecha_referencia = datos_limpios["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = datos_limpios.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (fecha_referencia - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("ValorTotal", "sum")
).reset_index()

print("Número de clientes:", rfm.shape[0])
rfm.head()



deberías obtener una tabla con alrededor de 4.300 clientes, cada uno con sus valores de `Recency` (en días), `Frequency` (número de facturas) y `Monetary` (gasto total). Este es el corazón del reto: pasamos de más de 390.000 filas de transacciones individuales a una sola fila por cliente que resume su comportamiento.


### Revisar la distribución de las variables RFM

Antes de aplicar el modelo de clustering, miramos cómo se distribuyen las tres variables.


In [ ]:

rfm[["Recency", "Frequency", "Monetary"]].describe()


In [ ]:

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))

rfm["Recency"].hist(bins=30, ax=ejes[0])
ejes[0].set_title("Recency (días)")

rfm["Frequency"].hist(bins=30, ax=ejes[1])
ejes[1].set_title("Frequency (facturas)")

rfm["Monetary"].hist(bins=30, ax=ejes[2])
ejes[2].set_title("Monetary (gasto total)")

plt.tight_layout()
plt.show()



es normal que `Frequency` y `Monetary` salgan muy sesgadas hacia la izquierda: la mayoría de los clientes compran pocas veces y gastan montos moderados, mientras que un grupo pequeño de clientes (probablemente compradores mayoristas) tiene frecuencias y montos mucho más altos que el resto. `Recency` suele tener una forma más repartida, con muchos clientes que compraron hace poco y otro grupo considerable que lleva meses sin comprar. Esta asimetría es la razón por la que, antes del K-Means, vamos a **normalizar** las variables — sin eso, `Monetary` dominaría completamente la distancia entre clientes solo por tener números mucho más grandes.


## Fase 5: Segmentación con aprendizaje no supervisado (K-Means)

### Normalizar las variables RFM

`StandardScaler` deja cada variable con media 0 y desviación estándar 1, para que ninguna domine el cálculo de distancias solo por su escala.


In [ ]:

scaler = StandardScaler()
rfm_escalado = scaler.fit_transform(rfm[["Recency", "Frequency", "Monetary"]])

print("Forma de los datos escalados:", rfm_escalado.shape)



`rfm_escalado` es un arreglo de NumPy con las mismas 4.300 filas y 3 columnas, pero ahora todas las variables están en la misma escala. Ya no vas a ver "días" o "dólares" — son valores estandarizados, que es lo que necesita el algoritmo K-Means para calcular distancias de forma justa entre las tres variables.


### Probar diferentes valores del número de clusters (método del codo)

No sabemos de antemano cuántos segmentos tiene sentido crear, así que probamos varios valores de *k* y observamos la "inercia" (qué tan compactos quedan los clusters) para cada uno.


In [ ]:

inercias = []
valores_k = range(1, 11)

for k in valores_k:
    modelo_kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    modelo_kmeans.fit(rfm_escalado)
    inercias.append(modelo_kmeans.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(valores_k, inercias, marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.grid()
plt.show()



la inercia siempre baja a medida que aumenta *k* (más clusters siempre ajustan mejor), pero buscamos el punto donde la curva deja de bajar de forma pronunciada y se vuelve más plana — ahí está el "codo". En datasets RFM como este, ese punto suele ubicarse entre **k = 4 y k = 5**. Ese es el número de segmentos que vamos a usar; si en tu gráfica el codo se ve en otro valor, ajusta `k` en la siguiente celda.


### Entrenar el modelo final y asignar segmentos

Usamos k = 4 como punto de partida razonable (ajusta este número según el codo que hayas observado).


In [ ]:

k_final = 4

modelo_kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10)
rfm["Segmento"] = modelo_kmeans.fit_predict(rfm_escalado)

rfm.head()



cada cliente ahora tiene una columna `Segmento` con un número de 0 a 3 (o hasta el valor de `k_final` que hayas elegido menos uno). Ese número no tiene un significado de negocio por sí solo — es solo una etiqueta de grupo. En la siguiente fase le vamos a dar sentido, describiendo el comportamiento típico de cada segmento.


## Fase 6: Interpretación de los segmentos

### Perfil promedio de cada segmento

Calculamos el promedio de Recency, Frequency y Monetary por segmento, junto con cuántos clientes y qué porcentaje de los ingresos totales representa cada uno.


In [ ]:

perfil_segmentos = rfm.groupby("Segmento").agg(
    Clientes=("CustomerID", "count"),
    Recency_promedio=("Recency", "mean"),
    Frequency_promedio=("Frequency", "mean"),
    Monetary_promedio=("Monetary", "mean"),
    Ingreso_total=("Monetary", "sum")
).round(1)

perfil_segmentos["Porcentaje_ingresos"] = (
    perfil_segmentos["Ingreso_total"] / perfil_segmentos["Ingreso_total"].sum() * 100
).round(1)

perfil_segmentos



esta tabla es la más importante del reto para la parte de negocio. Para leerla, compara las filas: el segmento con **Recency_promedio bajo, Frequency_promedio alto y Monetary_promedio alto** son tus clientes de mayor valor — probablemente representen una fracción pequeña de los clientes pero un porcentaje alto de los ingresos (patrón típico "80/20"). El segmento con **Recency_promedio muy alto** (muchos días sin comprar) y Frequency/Monetary bajos son clientes inactivos o en riesgo de abandono. Con estos cuatro números por segmento ya puedes proponerles una etiqueta descriptiva, por ejemplo: "Clientes de alto valor", "Clientes frecuentes", "Clientes ocasionales" e "Inactivos" — asígnalas según lo que veas en tu tabla.


### Visualizar distribución de clientes e ingresos por segmento


In [ ]:

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))

perfil_segmentos["Clientes"].plot(kind="bar", ax=ejes[0], color="steelblue")
ejes[0].set_title("Número de clientes por segmento")
ejes[0].set_xlabel("Segmento")
ejes[0].set_ylabel("Clientes")

perfil_segmentos["Porcentaje_ingresos"].plot(kind="bar", ax=ejes[1], color="darkorange")
ejes[1].set_title("Porcentaje de ingresos por segmento")
ejes[1].set_xlabel("Segmento")
ejes[1].set_ylabel("% de ingresos totales")

plt.tight_layout()
plt.show()



en un buen ejercicio de segmentación es común encontrar que el segmento más pequeño en número de clientes es el que concentra el mayor porcentaje de ingresos — esa es la evidencia visual de por qué vale la pena tratar a los clientes de forma diferenciada en vez de aplicarles a todos la misma estrategia.


## Fase 7: Modelo supervisado explicativo (árbol de decisión)

Ahora que tenemos los segmentos, entrenamos un **árbol de decisión** que aprenda a predecir el segmento de un cliente a partir de sus variables RFM. El objetivo no es maximizar la exactitud, sino obtener **reglas simples e interpretables** que expliquen qué hace que un cliente pertenezca a cada segmento.


In [ ]:

X_rfm = rfm[["Recency", "Frequency", "Monetary"]]
y_segmento = rfm["Segmento"]

X_train, X_test, y_train, y_test = train_test_split(
    X_rfm, y_segmento, test_size=0.2, random_state=42, stratify=y_segmento
)

arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_train, y_train)

y_pred = arbol.predict(X_test)
print("Exactitud del árbol:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred, zero_division=0))



como el árbol está aprendiendo a reconstruir los mismos segmentos que el propio K-Means ya definió a partir de estas tres variables, la exactitud suele ser muy alta (típicamente por encima de 0.90). Eso es normal y esperado aquí — no estamos prediciendo algo desconocido, sino explicando con reglas simples una segmentación que ya existe. Limitamos el árbol a `max_depth=3` a propósito, para que las reglas sean cortas y fáciles de explicar a alguien del negocio, en vez de buscar el árbol más preciso posible.

In [ ]:

plt.figure(figsize=(16, 8))
plot_tree(
    arbol,
    feature_names=["Recency", "Frequency", "Monetary"],
    class_names=[f"Segmento {c}" for c in sorted(y_segmento.unique())],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Árbol de decisión: reglas para clasificar el segmento de un cliente")
plt.show()



cada división del árbol es una regla legible, del tipo "si Monetary es mayor a cierto valor, entonces...". Sigue el árbol desde arriba hacia abajo: las primeras divisiones (las más cercanas a la raíz) son las variables que más separan a los segmentos — en datasets RFM casi siempre es `Monetary` o `Frequency` la que aparece primero, porque son las que más distinguen a un cliente de alto valor del resto. Estas reglas son las que puedes usar para explicarle a alguien no técnico, en un par de frases, qué hace que un cliente caiga en cada segmento.


## Fase 8: Conclusiones

En este notebook completamos el flujo analítico del reto:

- **EDA:** identificamos que el dataset tiene muchos valores faltantes en `CustomerID`, cancelaciones y outliers, típicos de datos transaccionales reales.
- **Preparación de datos:** filtramos cancelaciones, valores inválidos y clientes sin identificar.
- **RFM:** transformamos más de 390.000 transacciones en un perfil de comportamiento por cliente (Recency, Frequency, Monetary).
- **K-Means:** segmentamos a los clientes en grupos con comportamientos similares, eligiendo el número de clusters con el método del codo.
- **Interpretación:** describimos cada segmento en términos de negocio (número de clientes, gasto promedio, porcentaje de ingresos).
- **Árbol de decisión:** obtuvimos reglas simples que explican por qué un cliente pertenece a cada segmento.

**Próximo paso (fuera de este notebook):** el reto también pide un Producto Mínimo Viable — una aplicación (por ejemplo, con Streamlit) que ejecute este mismo flujo y muestre los resultados en un dashboard interactivo con los KPIs, gráficos y tablas que pide la Fase 8 del reto. Dime si quieres que te arme ese dashboard como siguiente entregable.
